In [38]:
import os
from dotenv import load_dotenv
load_dotenv()
import gc
import torch
import torch.nn.functional as F
import math
import numpy as np
import json
from glob import glob
from pathlib import Path
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, AutoModelForMaskedLM, BitsAndBytesConfig

import traceback

from huggingface_hub import login


In [39]:
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')

    CORPORA_FOLDER = "./corpora/"
    OUTPUT_FOLDER = "./perplexity_outputs/"

    os.makedirs(CORPORA_FOLDER, exist_ok = True)
    os.makedirs(OUTPUT_FOLDER, exist_ok = True)

except ModuleNotFoundError as e:
    hf_token = os.getenv("HF_TOKEN")
    CORPORA_FOLDER = "../../corpora/GeneratedCorpus/"
    OUTPUT_FOLDER = "../../perplexity_outputs/"

login(hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [40]:
def clean_memory():
    """Force RAM and VRAM to clean"""
    gc.collect()                # Clean not used python objects
    torch.cuda.empty_cache()    # Clean Pytorch alocation cache
    print("🧹 Memória limpa!")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Ambiente de Execução: {device.upper()}")

🚀 Ambiente de Execução: CUDA


In [41]:
def load_model_and_tokenizer(model_name:str, quantitize = False):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=False,
        trust_remote_code=True
    )

    bnb_config = None
    if quantitize:
        bnb_config = BitsAndBytesConfig(
            load_in_8bit=True,
            bnb_8bit_compute_dtype=torch.float16
        )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        trust_remote_code=True,
        #device_map="auto",
        dtype=torch.float16
    )

    model.to(device)
    model.eval()

    return tokenizer, model

In [42]:
def eval_causal_sliding(model, tokenizer, dataset, stride=512):
    results = {}

    max_len = model.config.max_position_embeddings

    # The perplexity is calculated for each text
    for label, texts in dataset.items():
        text_results = []

        for i, text in enumerate(texts):

            inputs = tokenizer(text, return_tensors="pt").to(model.device)
            num_tokens_text = inputs.input_ids.size(1)

            num_bytes = len(text.encode("utf-8"))
            num_words = len(text.split())

            total_nll = 0.0
            total_tokens = 0
            prev_end_window = 0

            # Sliding window
            for begin_window in range(0, num_tokens_text, stride):
                end_window = min(begin_window + max_len, num_tokens_text)

                # Since the windows overlap, this indicates how many new tokens
                # will be actually used to calculate the perplexity
                target_len = end_window - prev_end_window

                # Select only the tokens from the window
                input_ids = inputs.input_ids[:, begin_window:end_window]
                target_ids = input_ids.clone()

                # Loss will ignore labels with value -100
                target_ids[:, :-target_len] = -100

                with torch.no_grad():
                    outputs = model(input_ids, labels=target_ids)
                    neg_log_likelihood = outputs.loss

                total_nll += neg_log_likelihood.item() * target_len
                total_tokens += target_len
                prev_end_window = end_window

                if end_window == num_tokens_text:
                    break

            ppl = math.exp(total_nll / total_tokens)

            bpb = total_nll / (math.log(2) * num_bytes)

            tokens_per_word = total_tokens / num_words if num_words > 0 else 0

            text_results.append({
                "text_id": i,
                "num_tokens": total_tokens,
                "num_words": num_words,
                "num_bytes": num_bytes,
                "tokens_per_word": tokens_per_word,
                "ppl": ppl,
                "bpb": bpb
            })

        avg_ppl = sum(r["ppl"] for r in text_results) / len(text_results)
        avg_bpb = sum(r["bpb"] for r in text_results) / len(text_results)
        avg_tokens_per_word = sum(r["tokens_per_word"] for r in text_results) / len(text_results)

        results[label] = {
            "avg_ppl": avg_ppl,
            "avg_bpb": avg_bpb,
            "avg_tokens_per_word": avg_tokens_per_word,
            "texts": text_results
        }

    return results

In [43]:
def execute_experiment(model_name, model_url, quantitize, corpora_folder=CORPORA_FOLDER, output_folder=OUTPUT_FOLDER):
    
    print(f"Executing {model_name}")

    glob_path = os.path.join(corpora_folder, "*.json")
    print(glob_path)
    corpora_files = glob(glob_path)
    corpora_files.sort()
    
    print(corpora_files)

    all_results = dict()

    intermediate_results_folder = os.path.join(output_folder, model_name)
    os.makedirs(intermediate_results_folder, exist_ok=True) 

    try:
        
        tokenizer, model = load_model_and_tokenizer(model_url, quantitize)
        print("✅ Model and tokenizer loaded")

        for file_path in corpora_files:
            corpora_name = Path(file_path).stem

            print(f"⌛ Processing corpora: {corpora_name}")

            with open(file_path, "r", encoding="utf-8") as f:
                dataset = json.load(f)

            # Executing experiment
            results = eval_causal_sliding(model, tokenizer, dataset)


            # Saving intermediate results
            result_file = os.path.join(intermediate_results_folder, f"{model_name}-{corpora_name}.json")
            with open(result_file, 'w') as f:
                json.dump(results, f, indent=4, ensure_ascii=False)

            # Printing intermediate results
            print(f"\n{'Language':>10} | {'PPL':>10}")
            print(f"-"*25)

            for language in results.keys():
                print(f"{language:>10} | {results[language]["avg_ppl"]:>10.3f}")

            all_results[corpora_name] = results

            print("\n", "="*50, "\n")

        # Saving results to a single file
        results_file = f"{output_folder}/{model_name}.json"

        print("="*50)
        print("RESULT FILE: ", result_file)

        print(f"\nSaving results to: {results_file}")
        with open(results_file, 'w') as f:
            json.dump(all_results, f, indent=4, ensure_ascii=False)


    except Exception as e:
        print(f"\n\nErro: {e}\n\n")   
        traceback.print_exc()
     

    finally:
        del model
        del tokenizer
        clean_memory()

    # return all_results


In [45]:
BASE_CORPUS_PATH = "../../corpora/GeneratedCorpus/"
BASE_OUTPUT_PATH = "../../results2/"

EXPERIMENTS = {
    "corpora": [
        os.path.join(BASE_CORPUS_PATH, "GRAMMAR SWAP"),
        os.path.join(BASE_CORPUS_PATH, "LEXICAL SWAP"),
        os.path.join(BASE_CORPUS_PATH, "TRADUÇÃO"),
    ],
    "output": [
        os.path.join(BASE_OUTPUT_PATH, "GRAMMAR_SWAP"),
        os.path.join(BASE_OUTPUT_PATH, "LEXICAL_SWAP"),
        os.path.join(BASE_OUTPUT_PATH, "TRADUÇÃO"),
    ]
}

for dirs in EXPERIMENTS.values():
    for dir in dirs:
        os.makedirs(dir, exist_ok=True)

In [ ]:
models = {
    "Gemma": ("google/gemma-4-12B", False),
    "Qwen": ("Qwen/Qwen3.5-9B", False),
    "LLama": ("meta-llama/Meta-Llama-3-8B", False),
    "Ministral": ("mistralai/Ministral-3-8B-Base-2512", False),
    
    "Tucano": ("TucanoBR/Tucano-2b4", False),
    "Bode": ("recogna-nlp/bode-7b-alpaca-pt-br-no-peft", False),
    "Cabrita": ("22h/open-cabrita3b", False),
    "Sabia": ("maritaca-ai/sabia-7b", False),
    
    # "TinyGPT2": ("sshleifer/tiny-gpt2", False)
}

In [47]:
for (corpora, output) in zip(EXPERIMENTS["corpora"], EXPERIMENTS["output"]):
    for model_name, (model_url, quantization) in models.items():
        
        execute_experiment(
            model_name, 
            model_url, 
            quantization,
            corpora,
            output
        ) 

Executing TinyGPT2
../../corpora/GeneratedCorpus/GRAMMAR SWAP/*.json
['../../corpora/GeneratedCorpus/GRAMMAR SWAP/CAPACITAR.json', '../../corpora/GeneratedCorpus/GRAMMAR SWAP/COMPARTILHAR.json', '../../corpora/GeneratedCorpus/GRAMMAR SWAP/EXPLICAR.json']


Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model and tokenizer loaded
⌛ Processing corpora: CAPACITAR

  Language |        PPL
-------------------------
  ESPANHOL |  50287.284
   FRANCÊS |  50307.307
    GALEGO |  50281.430
    INGLÊS |  50338.768
  MANDARIM |  50306.387
 PB NATIVO |  50299.360
     RUSSO |  50305.296
     ÁRABE |  50282.439


⌛ Processing corpora: COMPARTILHAR

  Language |        PPL
-------------------------
   FRANCÊS |  50310.493
    INGLÊS |  50348.442
  MANDARIM |  50328.782
 PB NATIVO |  50315.806
     RUSSO |  50282.488
     ÁRABE |  50307.027


⌛ Processing corpora: EXPLICAR

  Language |        PPL
-------------------------
    INGLÊS |  50249.780
  MANDARIM |  50295.373
 PB NATIVO |  50264.233
     RUSSO |  50209.921
     ÁRABE |  50226.655


RESULT FILE:  ../../results2/GRAMMAR_SWAP/TinyGPT2/TinyGPT2-EXPLICAR.json

Saving results to: ../../results2/GRAMMAR_SWAP/TinyGPT2.json
🧹 Memória limpa!
Executing TinyGPT2
../../corpora/GeneratedCorpus/LEXICAL SWAP/*.json
['../../corpora/GeneratedCorpus/LEXI

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model and tokenizer loaded
⌛ Processing corpora: CAPACITAR

  Language |        PPL
-------------------------
  ESPANHOL |  50260.087
   FRANCÊS |  50216.083
    GALEGO |  50254.289
    INGLÊS |  50263.434
  MANDARIM |  50223.104
 PB NATIVO |  50299.360
     RUSSO |  50258.323
     ÁRABE |  50306.741


⌛ Processing corpora: COMPARTILHAR

  Language |        PPL
-------------------------
  ESPANHOL |  50300.090
   FRANCÊS |  50295.136
    GALEGO |  50285.080
    INGLÊS |  50254.303
  MANDARIM |  50260.388
 PB NATIVO |  50315.806
     RUSSO |  50290.622
     ÁRABE |  50287.829


⌛ Processing corpora: EXPLICAR

  Language |        PPL
-------------------------
  ESPANHOL |  50266.937
   FRANCÊS |  50300.358
    GALEGO |  50232.587
    INGLÊS |  50281.869
  MANDARIM |  50264.244
 PB NATIVO |  50264.233
     RUSSO |  50259.198
     ÁRABE |  50296.598


⌛ Processing corpora: EXPLORAR


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1504 > 1024). Running this sequence through the model will result in indexing errors



  Language |        PPL
-------------------------
  ESPANHOL |  50249.417
   FRANCÊS |  50238.647
    GALEGO |  50214.507
    INGLÊS |  50241.611
  MANDARIM |  50273.718
 PB NATIVO |  50236.540
     RUSSO |  50226.087
     ÁRABE |  50189.134


⌛ Processing corpora: FAZER

  Language |        PPL
-------------------------
  ESPANHOL |  50238.004
   FRANCÊS |  50250.102
    GALEGO |  50214.489
    INGLÊS |  50297.827
  MANDARIM |  50268.283
 PB NATIVO |  50212.153
     RUSSO |  50261.245
     ÁRABE |  50113.523


⌛ Processing corpora: RECOMENDAR

  Language |        PPL
-------------------------
  ESPANHOL |  50272.159
   FRANCÊS |  50252.910
    GALEGO |  50217.859
    INGLÊS |  50267.743
  MANDARIM |  50264.020
 PB NATIVO |  50252.137
     RUSSO |  50264.398
     ÁRABE |  50190.733


⌛ Processing corpora: RECRIAR

  Language |        PPL
-------------------------
  ESPANHOL |  50285.313
   FRANCÊS |  50235.830
    GALEGO |  50249.800
    INGLÊS |  50269.826
  MANDARIM |  50284.480
 PB